# Demostración del Componente NLP
Este notebook muestra ejemplos reales usando el componente NLP del proyecto.

In [28]:
# Instala dependencias en el kernel actual del notebook (si faltan)
%pip install pandas numpy joblib pyarrow scikit-learn xgboost tensorflow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached optree-0.19.1-cp313-cp313-win_amd64.whl.metadata (32 kB)
Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl (351.2 MB)
Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl (4.9 MB)
Using cached h5py-3.14.0-cp


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\cgalv\source\Python\AI\Transformers\.venv\Scripts\python.exe -m pip install --upgrade pip


In [96]:
# Demostracion real: explicabilidad y resumen de riesgo usando modelo ML entrenado
import sys
import importlib
from pathlib import Path

import numpy as np
import joblib

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import DATASET_PATH, MODELS_DIR, ML_SAMPLE_SIZE, PREDICTION_THRESHOLD

import src.ml.preprocess as preprocess_module
import src.nlp.nlp_component as nlp_module

preprocess_module = importlib.reload(preprocess_module)
nlp_module = importlib.reload(nlp_module)

load_processed = preprocess_module.load_processed
preprocess = preprocess_module.preprocess
NLPComponent = nlp_module.NLPComponent

MODEL_PATH = MODELS_DIR / "xgboost.pkl"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro el dataset en {DATASET_PATH}. "
        "Debes colocar el CSV en data/financial_data.csv"
    )

try:
    X_train, X_test, y_train, y_test, scaler = load_processed()
except FileNotFoundError:
    X_train, X_test, y_train, y_test, scaler = preprocess(sample_size=ML_SAMPLE_SIZE, save=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro el modelo en {MODEL_PATH}. "
        "Entrena primero con: python src/ml/train.py"
    )

ml_model = joblib.load(MODEL_PATH)

# Predicciones reales del modelo entrenado
y_prob = ml_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= PREDICTION_THRESHOLD).astype(int)

nlp = NLPComponent(feature_names=X_test.columns.tolist())

# Explicacion real sobre una transaccion realmente clasificada como fraude
idx_fraude = np.where((y_pred == 1))[0]
if len(idx_fraude) > 0:
    idx = idx_fraude[0]
    explicacion = nlp.explain_prediction(X_test.iloc[idx], ml_model)
    print(f"Ejemplo de explicacion para una transaccion clasificada como FRAUDE:\n{explicacion}\n")
else:
    print("No se detectaron fraudes en el set de prueba.")

# Resumen real del lote
resumen = nlp.summarize_batch(X_test, y_pred, y_prob)
print(f"Resumen del lote de prueba:\n{resumen}\n")

Ejemplo de explicacion para una transaccion clasificada como FRAUDE:
Explicación de la predicción:
Los siguientes atributos influyeron más en la decisión del modelo:
- origBalanceZero: valor=1.00, importancia=0.52
- errorBalanceOrig: valor=-0.32, importancia=0.45
- isHighRiskType: valor=1.00, importancia=0.01

Resumen del lote de prueba:
Se analizaron 40000 transacciones.
- 1644 fueron clasificadas como FRAUDE.
- 38356 como normales.
El riesgo promedio estimado fue de 4.12%.



## Ejemplo 2: Transacciones reales del set de prueba

Este ejemplo muestra transacciones reales del dominio financiero tomadas del set de prueba, junto con su probabilidad de fraude y nivel de riesgo estimado por el modelo entrenado.

In [97]:
# Análisis real de transacciones del set de prueba (sin simulación)
import pandas as pd

# Construir vista de análisis con predicciones reales
analysis_df = X_test.copy().reset_index(drop=True)
analysis_df["fraud_probability"] = y_prob
analysis_df["predicted_fraud"] = y_pred
analysis_df["actual_fraud"] = y_test.reset_index(drop=True)
analysis_df["risk_level"] = pd.cut(
    analysis_df["fraud_probability"],
    bins=[-0.01, 0.3, 0.7, 1.0],
    labels=["BAJO", "MEDIO", "ALTO"],
)

# Mostrar casos más riesgosos detectados por el modelo
cols = [
    "amount", "oldbalanceOrg", "newbalanceOrig", "isHighRiskType",
    "fraud_probability", "risk_level", "predicted_fraud", "actual_fraud",
]

print("Top 10 transacciones con mayor riesgo estimado:")
display(analysis_df.sort_values("fraud_probability", ascending=False)[cols].head(10))

print("\nTop 10 transacciones de bajo riesgo estimado:")
display(analysis_df.sort_values("fraud_probability", ascending=True)[cols].head(10))

Top 10 transacciones con mayor riesgo estimado:


,amount,oldbalanceOrg,newbalanceOrig,isHighRiskType,fraud_probability,risk_level,predicted_fraud,actual_fraud
39987,3.306002,0.716754,-0.285417,1,0.999997,ALTO,1,1
39968,10.751194,2.815837,-0.285417,1,0.999997,ALTO,1,1
28965,3.828199,0.863981,-0.285417,1,0.999997,ALTO,1,1
650,2.987219,0.626877,-0.285417,1,0.999997,ALTO,1,1
35865,5.472274,1.327509,-0.285417,1,0.999997,ALTO,1,1
28114,3.750961,0.842205,-0.285417,1,0.999997,ALTO,1,1
22471,6.056060,1.492100,-0.285417,1,0.999997,ALTO,1,1
29084,2.162872,0.394462,-0.285417,1,0.999997,ALTO,1,1
13079,10.078453,2.626165,-0.285417,1,0.999997,ALTO,1,1
8319,2.510644,0.492512,-0.285417,1,0.999997,ALTO,1,1



Top 10 transacciones de bajo riesgo estimado:


,amount,oldbalanceOrg,newbalanceOrig,isHighRiskType,fraud_probability,risk_level,predicted_fraud,actual_fraud
23959,-0.063972,-0.293764,-0.220403,0,9.143590e-07,BAJO,0,0
35030,-0.057178,-0.293335,-0.218036,0,9.582692e-07,BAJO,0,0
4559,0.006314,-0.294114,-0.200742,0,1.121675e-06,BAJO,0,0
17121,-0.271818,-0.294015,-0.285417,0,1.171356e-06,BAJO,0,0
26598,-0.271183,-0.294025,-0.285417,0,1.171356e-06,BAJO,0,0
34870,-0.267153,-0.293989,-0.285417,0,1.171356e-06,BAJO,0,0
6658,-0.271428,-0.293147,-0.285417,0,1.280106e-06,BAJO,0,0
6683,-0.270342,-0.293732,-0.285417,0,1.287749e-06,BAJO,0,0
37006,-0.266899,-0.292054,-0.285417,0,1.292045e-06,BAJO,0,0
38744,-0.270171,-0.293123,-0.285417,0,1.315867e-06,BAJO,0,0


## Explicabilidad y resumen de riesgo

En este ejemplo, el componente NLP genera explicaciones automáticas de por qué una transacción fue clasificada como fraude y un resumen textual del riesgo en el lote de prueba, usando los resultados reales del modelo ML entrenado sobre transacciones bancarias.

In [98]:
# Bloque explícito: explicabilidad y resumen de riesgo (ML real)
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Matriz de confusión (ML):")
print(cm)
print("\nResumen de errores:")
print(f"- True Positives (fraudes detectados): {tp}")
print(f"- False Negatives (fraudes no detectados): {fn}")
print(f"- False Positives (alertas falsas): {fp}")
print(f"- True Negatives (normales correctos): {tn}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Fraude"]))

# Mostrar ejemplos reales de errores para explicabilidad
analysis_errors = X_test.copy().reset_index(drop=True)
analysis_errors["actual_fraud"] = y_test.reset_index(drop=True)
analysis_errors["predicted_fraud"] = y_pred
analysis_errors["fraud_probability"] = y_prob
analysis_errors["error_type"] = "correct"
analysis_errors.loc[(analysis_errors["actual_fraud"] == 1) & (analysis_errors["predicted_fraud"] == 0), "error_type"] = "false_negative"
analysis_errors.loc[(analysis_errors["actual_fraud"] == 0) & (analysis_errors["predicted_fraud"] == 1), "error_type"] = "false_positive"

print("\nEjemplos de False Negatives (riesgo crítico):")
display(analysis_errors[analysis_errors["error_type"] == "false_negative"].sort_values("fraud_probability").head(5))

print("\nEjemplos de False Positives (fricción al usuario):")
display(analysis_errors[analysis_errors["error_type"] == "false_positive"].sort_values("fraud_probability", ascending=False).head(5))

Matriz de confusión (ML):
[[38348     9]
 [    8  1635]]

Resumen de errores:
- True Positives (fraudes detectados): 1635
- False Negatives (fraudes no detectados): 8
- False Positives (alertas falsas): 9
- True Negatives (normales correctos): 38348

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     38357
      Fraude       0.99      1.00      0.99      1643

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000


Ejemplos de False Negatives (riesgo crítico):


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceOrig,errorBalanceDest,origBalanceZero,amountToOrigRatio,isHighRiskType,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,actual_fraud,predicted_fraud,fraud_probability,error_type
36335,-1.594797,-0.101526,-0.295751,-0.285417,-0.301448,-0.278027,-0.073065,-0.144357,1,0.152479,1,False,True,False,False,False,1,0,0.000235,false_negative
27968,-1.574586,-0.236487,-0.295751,-0.285417,0.055680,0.021767,-0.252916,-0.144357,1,-0.057027,1,False,True,False,False,False,1,0,0.001055,false_negative
29114,-1.426372,-0.147164,-0.295751,-0.285417,-0.239609,-0.230969,-0.133883,-0.144357,1,0.081633,1,False,True,False,False,False,1,0,0.001196,false_negative
8,-1.554375,0.339254,-0.295751,-0.285417,-0.000284,0.095652,0.514327,-0.144357,1,0.836722,1,False,True,False,False,False,1,0,0.001282,false_negative
18058,-0.274342,0.072584,-0.295751,-0.285417,-0.314266,-0.251898,0.158957,-0.144357,1,0.422758,1,False,True,False,False,False,1,0,0.001662,false_negative



Ejemplos de False Positives (fricción al usuario):


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceOrig,errorBalanceDest,origBalanceZero,amountToOrigRatio,isHighRiskType,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,actual_fraud,predicted_fraud,fraud_probability,error_type
12370,-0.099180,-0.038250,-0.226183,-0.285417,-0.296307,-0.259498,-0.317565,-0.144357,1,-0.132693,1,False,True,False,False,False,0,1,0.999790,false_positive
11151,0.055772,-0.117136,-0.248400,-0.285417,1.796194,1.652211,-0.317676,-0.144357,1,-0.132693,1,False,True,False,False,False,0,1,0.999611,false_positive
29740,0.392622,-0.232069,-0.280815,-0.285417,-0.019431,-0.046510,-0.317624,-0.144357,1,-0.132693,1,False,True,False,False,False,0,1,0.998769,false_positive
36261,-1.419635,1.288302,-0.295751,-0.285417,-0.314266,0.195011,1.779047,-1.339495,1,2.309974,1,False,False,False,False,True,0,1,0.944631,false_positive
15307,-1.621745,0.007137,-0.295751,-0.285417,-0.314266,-0.266161,0.071742,-0.144357,1,0.321162,1,False,True,False,False,False,0,1,0.923607,false_positive


## Explicabilidad con modelo Deep Learning

A continuación se muestra cómo el componente NLP puede generar explicaciones y resúmenes usando el modelo de Deep Learning entrenado para clasificación de fraude.

In [99]:
# Demostracion real con modelo Deep Learning (sin simulacion)
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import (
    DATASET_PATH,
    MODELS_DIR,
    REPORTS_DIR,
    ML_SAMPLE_SIZE,
    PREDICTION_THRESHOLD,
)

import src.ml.preprocess as preprocess_module
from src.nlp.nlp_component import NLPComponent

preprocess_module = importlib.reload(preprocess_module)
load_processed = preprocess_module.load_processed
preprocess = preprocess_module.preprocess

DL_MODEL_PATH = MODELS_DIR / "dl_final_model.keras"
DL_EXAMPLES_PATH = REPORTS_DIR / "dl_prediction_examples.csv"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro el dataset en {DATASET_PATH}. "
        "Debes colocar el CSV en data/financial_data.csv"
    )

# Preferencia: inferencia directa del modelo DL entrenado
try:
    import tensorflow as tf
    tf_available = True
except ModuleNotFoundError:
    tf_available = False

if tf_available and DL_MODEL_PATH.exists():
    try:
        X_train, X_test, y_train, y_test, scaler = load_processed()
    except FileNotFoundError:
        X_train, X_test, y_train, y_test, scaler = preprocess(sample_size=ML_SAMPLE_SIZE, save=True)

    dl_model = tf.keras.models.load_model(DL_MODEL_PATH)
    dl_y_prob = dl_model.predict(X_test.values.astype("float32")).ravel()
    dl_y_pred = (dl_y_prob >= PREDICTION_THRESHOLD).astype(int)
    nlp = NLPComponent(feature_names=X_test.columns.tolist())
else:
    # Fallback real: usar predicciones guardadas del entrenamiento DL real
    if not DL_EXAMPLES_PATH.exists():
        missing_reason = "TensorFlow no disponible" if not tf_available else "modelo DL no encontrado"
        raise FileNotFoundError(
            f"No se pudo ejecutar inferencia DL ({missing_reason}) y no existe {DL_EXAMPLES_PATH}. "
            "Ejecuta: python src/dl/train_dl.py"
        )

    examples = pd.read_csv(DL_EXAMPLES_PATH)
    dl_y_prob = examples["fraud_probability"].to_numpy()
    dl_y_pred = examples["predicted"].to_numpy().astype(int)
    feature_cols = [
        c for c in examples.columns
        if c not in ["actual", "predicted", "fraud_probability", "result_type"]
    ]
    X_test = examples[feature_cols]
    nlp = NLPComponent(feature_names=X_test.columns.tolist())

# Explicacion y resumen sobre resultados DL reales
dl_idx_fraude = np.where((dl_y_pred == 1))[0]
if len(dl_idx_fraude) > 0:
    idx = dl_idx_fraude[0]
    top_features = X_test.iloc[idx].abs().sort_values(ascending=False).head(3)
    explicacion = "\n".join([f"- {f}: valor={v:.2f}" for f, v in top_features.items()])
    print(f"Explicacion (DL) para una transaccion clasificada como FRAUDE:\n{explicacion}\n")
else:
    print("No se detectaron fraudes en el set de prueba (DL).")

dl_resumen = nlp.summarize_batch(X_test, dl_y_pred, dl_y_prob)
print(f"Resumen del lote de prueba (DL):\n{dl_resumen}\n")

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Explicacion (DL) para una transaccion clasificada como FRAUDE:
- origBalanceZero: valor=1.00
- isHighRiskType: valor=1.00
- type_CASH_OUT: valor=1.00

Resumen del lote de prueba (DL):
Se analizaron 40000 transacciones.
- 2059 fueron clasificadas como FRAUDE.
- 37941 como normales.
El riesgo promedio estimado fue de 5.44%.

